# ==============================================================================
# UNIVERSIDAD DE LA SABANA - EICEA
# ASIGNATURA: Programación y Decisiones (2026-1)
## PROYECTO: Cafeteria_U_Sabana
## CLASE 6
## TEMA: POO Avanzada (Encapsulamiento, Herencia, Polimorfismo e Integración)
### Fecha: 11/03/2026
# =============================================================================

In [4]:
# ==============================================================================
# UNIVERSIDAD DE LA SABANA - EICEA
# ASIGNATURA: Programación y Decisiones (2026-1)
# PROYECTO: Cafeteria_U_Sabana
# TEMA: POO Avanzada (Encapsulamiento, Herencia, Polimorfismo e Integración)
# ==============================================================================

# ==============================================================================
# PASO 1: ENCAPSULAMIENTO (El Principio de Seguridad)
# ==============================================================================
# Creamos la Superclase (Clase Padre). Esta será nuestra futura tabla 'dim_productos' en SQL.
class Producto:
    # __init__ se usa para inicializar los atributos de cada producto. Es el constructor de la clase.
    # self se usa para referirse a la instancia actual del objeto. Es como decir "este producto específico".
    def __init__(self, nombre, precio, stock):
        
        # self.nombre significa que cada producto tendrá su propio nombre. 
        # Es un atributo público porque no tiene '__'.
        self.nombre = nombre
        
        # ENCAPSULAMIENTO: Usamos '__' para hacer los atributos PRIVADOS.
        # Evitamos que alguien haga: producto.__precio = -5000 (Sanity Check)
        self.__precio = precio 
        self.__stock = stock   
        
    # GETTERS: Métodos seguros para LEER los datos privados

    # get_precio significa "dame el precio", no "cambia el precio". Es solo lectura.
    def get_precio(self):
        return self.__precio

    # get_stock se utiliza para verificar cuánto stock queda, pero no para modificarlo directamente.    
    def get_stock(self):
        return self.__stock
        
    # SETTERS: Métodos seguros para MODIFICAR los datos privados con validación
    def reducir_stock(self, cantidad):

        # Validamos que no vendan más de lo que hay, ni cantidades negativas

        if cantidad > 0 and cantidad <= self.__stock:
            
            # Reducimos el stock de forma segura. 
            # Usamos self.__stock para acceder al atributo privado dentro de la clase.
            self.__stock -= cantidad 
            
            # return se usa para indicar el resultado de la función. 
            # En este caso, devuelve True si la reducción fue exitosa, o False si no se pudo realizar por falta de stock.
            return True # Venta exitosa
        return False # Venta fallida por falta de stock


# ==============================================================================
# PASO 2: HERENCIA (El Principio DRY - Don't Repeat Yourself)
# ==============================================================================
# Creamos Clases Hijo. Heredan 'nombre', '__precio' y '__stock' del Padre.

class Bebida(Producto):
    def __init__(self, nombre, precio, stock, tamano):
        
        # super() llama al constructor de la clase Padre (Producto).
        # super().__init__ significa "Ejecuta el código de Producto.__init__ para inicializar nombre, precio y stock".
        super().__init__(nombre, precio, stock) 
        
        # self.tamano se usa para almacenar el tamaño específico de la bebida (Pequeño, Mediano, Grande). 
        # Es un atributo exclusivo de Bebida.
        self.tamano = tamano
        
    # ==========================================================================
    # PASO 3: POLIMORFISMO (Mismo método, diferente comportamiento)
    # ==========================================================================
    # Las bebidas preparadas pagan 8% de Impoconsumo en Colombia.
    def calcular_impuesto(self):
        
        # get_precio() se usa para acceder al precio privado de forma segura.
        # get_precio() genera error si intentamos acceder a __precio directamente desde fuera de la clase, pero aquí dentro es perfectamente válido.
        
        # return se usa para enviar el resultado de la función al lugar donde se llamó. En este caso, devuelve el valor del impuesto calculado. 
        return self.get_precio() * 0.08 


class Snack(Producto):
    def __init__(self, nombre, precio, stock, gramos):
        super().__init__(nombre, precio, stock)
        self.gramos = gramos # Atributo exclusivo de Snack
        
    # POLIMORFISMO: El Snack también tiene 'calcular_impuesto', pero la matemática cambia.
    # Es Polimorfismo porque el mismo método 'calcular_impuesto' se comporta diferente según si es una Bebida o un Snack. 
    # Python lo maneja automáticamente según el tipo de objeto que se le pase.

    # Los snacks procesados pagan 19% de IVA.
    def calcular_impuesto(self):
        return self.get_precio() * 0.19 


# ==============================================================================
# PASO 4: INTEGRACIÓN (Ecosistema de Objetos)
# ==============================================================================
# Esta clase será nuestra futura tabla 'fact_ventas' en la base de datos SQL.
class CarritoDeCompras:
    
    # __init__(self): se usa para inicializar los atributos del carrito. 
    # Cada carrito tendrá su propia lista de items, subtotal e impuestos.
    # Solo se usa (self) porque no necesitamos pasar ningún dato externo al crear el carrito.
    def __init__(self):
        
        # Se usa [] para crear una lista vacía, y {} para crear un diccionario vacío y () para crear una tupla vacía.
        
        self.items = [] # Almacenará los OBJETOS Producto. Se usa [] porque es una colección ordenada y mutable, es decir, se puede modificar después de su creación.
        self.subtotal = 0.0 # Se usa para acumular el valor total de los productos sin impuestos. Es un número decimal, por eso se inicializa con 0.0.
        self.total_impuestos = 0.0 # Se usa para acumular el valor total de los impuestos. También es un número decimal.

    # Ahora el carrito no recibe textos sueltos, recibe OBJETOS completos de tipo Producto.
    def agregar_producto(self, producto, cantidad):
        
        # 1. Intentamos reducir el stock (Encapsulamiento en acción)
        # Encapsulamiento significa que el carrito no puede modificar el stock directamente, sino que debe usar el método seguro 'reducir_stock' del producto.
        
        # Usamos if / else porque reducir_stock devuelve True o False dependiendo de si la reducción fue exitosa o no. 
        # Esto nos permite manejar el flujo de la compra de forma segura.
        if producto.reducir_stock(cantidad):
            
            # 2. Agregamos el objeto a la lista
            
            # .append se usa para agregar un nuevo elemento al final de la lista 'items'. 
            # En este caso, agregamos un diccionario que contiene el producto y la cantidad comprada.

            self.items.append({
                "producto": producto, # producto es el OBJETO completo, no solo su nombre o precio. Esto nos permite acceder a todos sus métodos y atributos. 
                "cantidad": cantidad # cantidad es un número entero que indica cuántas unidades del producto se están comprando.
            })
            
            # 3. Cálculos financieros leyendo los métodos del objeto
            # producto.get_precio() se usa para obtener el precio del producto de forma segura, sin acceder al atributo privado __precio directamente.
            precio_base = producto.get_precio() * cantidad
            
            # POLIMORFISMO EN ACCIÓN: Python sabe automáticamente si aplicar 8% o 19% 
            # dependiendo de si el objeto es Bebida o Snack.
            # Es Polimorfismo porque el mismo método 'calcular_impuesto' se comporta diferente según el tipo de producto que se le pase (Bebida o Snack).
            # producto.calcular_impuesto() se usa para calcular el impuesto del producto, 
                # y luego se multiplica por la cantidad para obtener el impuesto total de esa línea de compra.
            impuesto = producto.calcular_impuesto() * cantidad

            self.subtotal += precio_base # Aquí se acumula el subtotal sumando el precio base de cada producto comprado.
            self.total_impuestos += impuesto # Aquí se acumulan los impuestos sumando el impuesto de cada producto comprado.
            
            print(f"✅ Agregado: {cantidad}x {producto.nombre} al carrito.")
        else:
            print(f"❌ Error: Stock insuficiente para {producto.nombre}.")

    # generar_factura(self) se usa para imprimir el resumen de la compra, mostrando cada producto, su cantidad, el subtotal, los impuestos y el total a pagar.
    def generar_factura(self):
        print("\n" + "="*45)
        print("🧾 FACTURA ELECTRÓNICA - CAFETERÍA U. SABANA")
        print("="*45)
        
        # Usamos if / else para manejar el caso de un carrito vacío, evitando errores al intentar imprimir productos que no existen.
        # Usamos len(self.items) para verificar cuántos productos hay en el carrito. Si es 0, mostramos un mensaje indicando que el carrito está vacío.
        if len(self.items) == 0:
            print("El carrito está vacío.")
        else:
            # Usamos un bucle for para iterar sobre cada item en la lista 'items' del carrito.
            for item in self.items:
                
                prod = item["producto"] # Extraemos el OBJETO. Esto nos permite acceder a sus métodos y atributos, como prod.get_precio() o prod.nombre.
                cant = item["cantidad"] # Extraemos la cantidad comprada de ese producto. Esto nos permite calcular el total de esa línea multiplicando el precio por la cantidad.
                
                # Usamos el getter para leer el precio privado
                total_linea = prod.get_precio() * cant
                
                # ljust(20) se usa para alinear el nombre del producto a la izquierda y reservar un espacio de 20 caracteres, lo que mejora la legibilidad de la factura.
                # prod.nombre se usa para mostrar el nombre del producto en la factura. Es un atributo público, por eso se puede acceder directamente sin necesidad de un getter.
                print(f"🛒 {prod.nombre.ljust(20)} (x{cant}) : ${total_linea:,.2f}")

            total_pagar = self.subtotal + self.total_impuestos
            
            print("-" * 45)
            print(f"Subtotal:      ${self.subtotal:,.2f} COP")
            print(f"Impuestos:     ${self.total_impuestos:,.2f} COP")
            print(f"TOTAL A PAGAR: ${total_pagar:,.2f} COP")
        print("="*45 + "\n")


# ==============================================================================
# PASO 5: EJECUCIÓN Y PRUEBAS (El Momento de la Verdad)
# ==============================================================================
if __name__ == "__main__":
    print("Iniciando Sistema Cafeteria_U_Sabana...\n")

    # 1. Creamos nuestro inventario instanciando las Clases Hijo
    cafe_tostao = Bebida(nombre="Café de Origen Tostao", precio=5000, stock=50, tamano="Mediano")
    chocolate_jet = Snack(nombre="Chocolatina Jet", precio=1200, stock=100, gramos=12)

    # 2. Inicializamos el sistema de ventas
    caja_registradora = CarritoDeCompras()

    # 3. Simulamos transacciones (El Carrito interactúa con los Objetos)
    caja_registradora.agregar_producto(cafe_tostao, 2)    # Compra 2 cafés
    caja_registradora.agregar_producto(chocolate_jet, 5)  # Compra 5 chocolatinas
    
    # Intento de compra masiva para probar el encapsulamiento de stock
    caja_registradora.agregar_producto(cafe_tostao, 100)  # Debe dar error (Solo hay 48 restantes)

    # 4. Generamos el reporte final
    caja_registradora.generar_factura()

Iniciando Sistema Cafeteria_U_Sabana...

✅ Agregado: 2x Café de Origen Tostao al carrito.
✅ Agregado: 5x Chocolatina Jet al carrito.
❌ Error: Stock insuficiente para Café de Origen Tostao.

🧾 FACTURA ELECTRÓNICA - CAFETERÍA U. SABANA
🛒 Café de Origen Tostao (x2) : $10,000.00
🛒 Chocolatina Jet      (x5) : $6,000.00
---------------------------------------------
Subtotal:      $16,000.00 COP
Impuestos:     $1,940.00 COP
TOTAL A PAGAR: $17,940.00 COP



# Taller Práctico en Clase (Grupos de 3)

## Título: Expandiendo el Ecosistema: Módulo de Clientes

## El Reto:
El sistema de la cafetería necesita registrar a los compradores.

- Creen una clase Padre llamada Cliente (Atributos: nombre, id).

- Creen dos clases Hijo: Estudiante y Profesor.

- Polimorfismo: Creen un método llamado obtener_descuento().

    Si es Estudiante, retorna 0.10 (10% de descuento).

    Si es Profesor, retorna 0.05 (5% de descuento).

- Conecten al cliente con el CarritoDeCompras para que el total a pagar aplique este descuento.


In [ ]:
# ==============================================================================
# TALLER EN CLASE: MÓDULO DE CLIENTES (Herencia y Polimorfismo)
# ==============================================================================

# 1. CLASE PADRE
class Cliente:
    def __init__(self, nombre, id_cliente):
        self.nombre = nombre
        self.id_cliente = id_cliente

    # Método genérico que será sobreescrito por las clases hijos
    def obtener_descuento(self):
        return 0.0 # 0% de descuento por defecto

# 2. CLASES HIJO
class Estudiante(Cliente):
    def __init__(self, nombre, id_cliente):
        super().__init__(nombre, id_cliente)
        
    # POLIMORFISMO: El estudiante tiene 10% de descuento
    def obtener_descuento(self):
        return 0.10 

class Profesor(Cliente):
    def __init__(self, nombre, id_cliente):
        super().__init__(nombre, id_cliente)
        
    # POLIMORFISMO: El profesor tiene 5% de descuento
    def obtener_descuento(self):
        return 0.05


# 3. PRUEBAS 
if __name__ == "__main__":
    
    print("Iniciando Pruebas de Cliente...\n")

    # 1. Creamos instancias de los clientes
    estudiante1 = Estudiante("Ana", "E001")
    profesor1 = Profesor("Carlos", "P001")

    # 2. Probamos el método polimórfico
    print(f"Descuento para {estudiante1.nombre}: {estudiante1.obtener_descuento() * 100:.0f}%")
    print(f"Descuento para {profesor1.nombre}: {profesor1.obtener_descuento() * 100:.0f}%")



Iniciando Pruebas de Cliente...

Descuento para Ana: 10%
Descuento para Carlos: 5%


# Modificaciones al CarritoDeCompras (Integración)

- Para que el carrito sepa a quién le está vendiendo y le aplique el descuento, debes reemplazar tu clase CarritoDeCompras actual por esta versión actualizada.

### ¿Qué cambió?

    El constructor __init__ ahora exige un objeto Cliente.

    El método generar_factura lee el método obtener_descuento() del cliente y hace la matemática final.

### Modificación a la Ejecución (Paso 5)

- Finalmente, al final de tu archivo (en la sección de pruebas), debes crear un cliente y pasarlo al carrito.

In [ ]:
# ==============================================================================
# PASO 4: INTEGRACIÓN (Ecosistema de Objetos) - VERSIÓN ACTUALIZADA
# ==============================================================================
class CarritoDeCompras:
    # MODIFICACIÓN 1: El carrito ahora recibe un OBJETO Cliente al crearse
    def __init__(self, cliente):
        self.cliente = cliente # Almacenamos el objeto cliente dentro del carrito para poder acceder a su método de descuento más adelante.
        self.items =[] 
        self.subtotal = 0.0
        self.total_impuestos = 0.0

    def agregar_producto(self, producto, cantidad):
        if producto.reducir_stock(cantidad):
            self.items.append({
                "producto": producto, 
                "cantidad": cantidad
            })
            
            precio_base = producto.get_precio() * cantidad
            impuesto = producto.calcular_impuesto() * cantidad

            self.subtotal += precio_base
            self.total_impuestos += impuesto
            
            print(f"✅ Agregado: {cantidad}x {producto.nombre} al carrito.")
        else:
            print(f"❌ Error: Stock insuficiente para {producto.nombre}.")

    def generar_factura(self):
        print("\n" + "="*45)
        print("🧾 FACTURA ELECTRÓNICA - CAFETERÍA U. SABANA")

        # MODIFICACIÓN 2: Imprimimos los datos del cliente
        
        print(f"👤 Cliente: {self.cliente.nombre} | ID: {self.cliente.id_cliente}")
        print("="*45)
        
        if len(self.items) == 0:
            print("El carrito está vacío.")
        else:
            for item in self.items:
                prod = item["producto"] 
                cant = item["cantidad"]
                total_linea = prod.get_precio() * cant
                print(f"🛒 {prod.nombre.ljust(20)} (x{cant}) : ${total_linea:,.2f}")

            # --- LÓGICA FINANCIERA Y DESCUENTOS (POLIMORFISMO) ---
            total_bruto = self.subtotal + self.total_impuestos
            
            # Aquí ocurre la magia del Polimorfismo: Python calculará 10% o 5% 
            # dependiendo de si el objeto inyectado fue Estudiante o Profesor.
            porcentaje_desc = self.cliente.obtener_descuento()
            valor_descuento = total_bruto * porcentaje_desc
            total_pagar = total_bruto - valor_descuento
            
            print("-" * 45)
            print(f"Subtotal:      ${self.subtotal:,.2f} COP")
            print(f"Impuestos:     ${self.total_impuestos:,.2f} COP")
            
            # Solo mostramos la línea de descuento si el cliente tiene uno
            if porcentaje_desc > 0:
                print(f"Descuento ({(porcentaje_desc*100):.0f}%): -${valor_descuento:,.2f} COP")
                
            print(f"TOTAL A PAGAR: ${total_pagar:,.2f} COP")
        print("="*45 + "\n")


# ==============================================================================
# PASO 5: EJECUCIÓN - VERSIÓN FINAL CON CLIENTES (Inyección de Dependencias)
# ==============================================================================
if __name__ == "__main__":
    print("Iniciando Sistema Cafeteria_U_Sabana...\n")

    # 1. Creamos nuestro inventario
    cafe_tostao = Bebida(nombre="Café de Origen Tostao", precio=5000, stock=50, tamano="Mediano")
    chocolate_jet = Snack(nombre="Chocolatina Jet", precio=1200, stock=100, gramos=12)

    # 2. Creamos al cliente (Prueba cambiando 'Estudiante' por 'Profesor' para ver cómo cambia el descuento)
    cliente_actual_1 = Estudiante(nombre="Diego Zuluaga", id_cliente="00012345") # Prueba con un estudiante para ver el descuento del 10%
    cliente_actual_2 = Profesor(nombre="Yeimy Castillo", id_cliente="00067890") # Prueba con un profesor para ver el cambio en el descuento

    # 3. Inicializamos el sistema de ventas INYECTANDO al cliente
    caja_registradora_1 = CarritoDeCompras(cliente=cliente_actual_1) # Inyectamos el cliente estudiante
    caja_registradora_2 = CarritoDeCompras(cliente=cliente_actual_2) # Inyectamos el cliente profesor

    # 4. Simulamos transacciones
    caja_registradora_1.agregar_producto(cafe_tostao, 2)    
    caja_registradora_2.agregar_producto(chocolate_jet, 5)  

    # 5. Generamos el reporte final
    caja_registradora_1.generar_factura()
    caja_registradora_2.generar_factura()

"""
Concepto Clave: Polimorfismo

Al momento de ejecutar self.cliente.obtener_descuento(), 
la clase CarritoDeCompras no sabe si el cliente es un Estudiante o un Profesor, 
y no le importa. Solo sabe que es un "Cliente" 
y que debe aplicarle el descuento que corresponda. 
Eso es el verdadero poder del Polimorfismo en la arquitectura de software.
"""

Iniciando Sistema Cafeteria_U_Sabana...

✅ Agregado: 2x Café de Origen Tostao al carrito.
✅ Agregado: 5x Chocolatina Jet al carrito.

🧾 FACTURA ELECTRÓNICA - CAFETERÍA U. SABANA
👤 Cliente: Diego Zuluaga | ID: 00012345
🛒 Café de Origen Tostao (x2) : $10,000.00
---------------------------------------------
Subtotal:      $10,000.00 COP
Impuestos:     $800.00 COP
Descuento (10%): -$1,080.00 COP
TOTAL A PAGAR: $9,720.00 COP


🧾 FACTURA ELECTRÓNICA - CAFETERÍA U. SABANA
👤 Cliente: Yeimy Castillo | ID: 00067890
🛒 Chocolatina Jet      (x5) : $6,000.00
---------------------------------------------
Subtotal:      $6,000.00 COP
Impuestos:     $1,140.00 COP
Descuento (5%): -$357.00 COP
TOTAL A PAGAR: $6,783.00 COP

